<a href="https://colab.research.google.com/github/xiyuan1avery/ma2288/blob/main/notebooks/02_extract_hidden_states.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets accelerate

In [2]:
import random
import numpy as np
import torch

# 固定随机种子，让实验尽可能可以重复
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 自动选择 GPU 或 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "distilgpt2"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# 把模型放到 GPU
model = model.to(device)

# 切换到评估模式，关闭 dropout 等训练时的随机操作
model.eval()

# 冻结语言模型参数
for parameter in model.parameters():
    parameter.requires_grad = False

print("Model loaded successfully.")
print("Hidden dimension:", model.config.n_embd)
print("Number of layers:", model.config.n_layer)
print("Number of parameters:", sum(p.numel() for p in model.parameters()))

Loading tokenizer...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully.
Hidden dimension: 768
Number of layers: 6
Number of parameters: 81912576


In [4]:
text = "The scientist opened the notebook and carefully recorded the result."

inputs = tokenizer(text, return_tensors="pt")

# 把输入也移动到 GPU
input_ids = inputs["input_ids"].to(device)
attention_mask = inputs["attention_mask"].to(device)

tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

print("Original text:")
print(text)

print("\nToken IDs:")
print(input_ids[0].tolist())

print("\nTokens:")
print(tokens)

print("\nNumber of tokens:")
print(len(tokens))

Original text:
The scientist opened the notebook and carefully recorded the result.

Token IDs:
[464, 11444, 4721, 262, 20922, 290, 7773, 6264, 262, 1255, 13]

Tokens:
['The', 'Ġscientist', 'Ġopened', 'Ġthe', 'Ġnotebook', 'Ġand', 'Ġcarefully', 'Ġrecorded', 'Ġthe', 'Ġresult', '.']

Number of tokens:
11


In [5]:
with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        output_hidden_states=True,
        return_dict=True,
    )

all_hidden_states = outputs.hidden_states
final_hidden_states = all_hidden_states[-1]

print("Number of hidden-state tensors:", len(all_hidden_states))
print("Final hidden-state shape:", final_hidden_states.shape)

Number of hidden-state tensors: 7
Final hidden-state shape: torch.Size([1, 11, 768])


In [6]:
print(f"{'Position':<10} {'Token':<20} {'Hidden norm':<15}")
print("-" * 50)

for position, token in enumerate(tokens):
    hidden_vector = final_hidden_states[0, position]
    hidden_norm = torch.linalg.vector_norm(hidden_vector).item()

    print(f"{position:<10} {repr(token):<20} {hidden_norm:<15.4f}")

Position   Token                Hidden norm    
--------------------------------------------------
0          'The'                76.1122        
1          'Ġscientist'         151.7086       
2          'Ġopened'            140.8430       
3          'Ġthe'               158.4968       
4          'Ġnotebook'          186.0373       
5          'Ġand'               192.3977       
6          'Ġcarefully'         195.9913       
7          'Ġrecorded'          170.1248       
8          'Ġthe'               179.0534       
9          'Ġresult'            149.3936       
10         '.'                  173.7315       


In [7]:
# 选择一个中间位置
t = min(4, input_ids.shape[1] - 1)

# 从完整句子中读取位置 t 的 hidden state
hidden_from_full_sequence = final_hidden_states[:, t, :]

# 只保留从开头到位置 t 的 prefix
prefix_input_ids = input_ids[:, : t + 1]
prefix_attention_mask = attention_mask[:, : t + 1]

with torch.no_grad():
    prefix_outputs = model(
        input_ids=prefix_input_ids,
        attention_mask=prefix_attention_mask,
        output_hidden_states=True,
        return_dict=True,
    )

hidden_from_prefix = prefix_outputs.hidden_states[-1][:, -1, :]

difference = torch.max(
    torch.abs(hidden_from_full_sequence - hidden_from_prefix)
).item()

print("Selected position:", t)
print("Selected token:", tokens[t])
print("Maximum absolute difference:", difference)

Selected position: 4
Selected token: Ġnotebook
Maximum absolute difference: 1.52587890625e-05


In [8]:
# logits 的形状也是 [batch, sequence length, vocabulary size]
logits = outputs.logits

# 取最后一个位置对下一个 token 的预测
next_token_logits = logits[0, -1]

# 转换成概率
next_token_probabilities = torch.softmax(next_token_logits, dim=-1)

# 找出概率最高的五个 token
top_probabilities, top_token_ids = torch.topk(
    next_token_probabilities,
    k=5,
)

print("Model's top-5 predictions for the next token:\n")

for probability, token_id in zip(top_probabilities, top_token_ids):
    predicted_text = tokenizer.decode([token_id.item()])
    print(f"{repr(predicted_text):<20} probability = {probability.item():.4f}")

Model's top-5 predictions for the next token:

'\n'                 probability = 0.3065
' The'               probability = 0.1142
' He'                probability = 0.1049
' It'                probability = 0.0403
' "'                 probability = 0.0267


In [9]:
import gc

gc.collect()
torch.cuda.empty_cache()

allocated_gb = torch.cuda.memory_allocated() / 1024**3
reserved_gb = torch.cuda.memory_reserved() / 1024**3

print(f"Allocated GPU memory: {allocated_gb:.3f} GB")
print(f"Reserved GPU memory: {reserved_gb:.3f} GB")

Allocated GPU memory: 0.319 GB
Reserved GPU memory: 0.344 GB
